# MASLD-Bench · 02 — Frontier Evaluation (Drive-synced edition)



Re-runs/extends the frontier bracket from your synced prompts.

Protocol: frozen ANALYSIS_PLAN_v1 — zero-shot, greedy, single-pass, 300/task,

max_tokens 200, reasoning disabled.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

SEARCH_ROOTS = ['/content/drive/Othercomputers/My Mac', '/content/drive/MyDrive']

def find_dir(target_name, search_roots=None, depth=6):
    if search_roots is None:
        search_roots = SEARCH_ROOTS
    matches = []
    for root in search_roots:
        direct = os.path.join(root, target_name)
        if os.path.isdir(direct):
            matches.append(direct)
    if matches:
        return matches
    def walk(path, d):
        if d > depth:
            return
        try:
            for entry in os.scandir(path):
                if entry.is_dir(follow_symlinks=False):
                    if entry.name == target_name:
                        matches.append(entry.path)
                    walk(entry.path, d + 1)
        except (PermissionError, OSError):
            pass
    for root in search_roots:
        if os.path.exists(root):
            walk(root, 0)
    return matches

cands = find_dir('masld-llm')
MASLD_LLM = min(cands, key=len) if cands else None
assert MASLD_LLM, 'masld-llm not found under Othercomputers/My Mac or MyDrive'
print('MASLD_LLM =', MASLD_LLM)

def is_writable(path):
    try:
        probe = os.path.join(path, '.write_test')
        open(probe, 'w').close(); os.remove(probe)
        return True
    except Exception:
        return False

RUNS = (os.path.join(MASLD_LLM, 'results', 'frontier_runs')
        if is_writable(os.path.join(MASLD_LLM, 'results'))
        else '/content/drive/MyDrive/masld-llm-outputs/frontier_runs')
os.makedirs(RUNS, exist_ok=True)
print('RUNS =', RUNS)

In [ ]:
# copy the runner + scorer from the synced scripts (they carry all fixes)
import shutil
shutil.copy(f'{MASLD_LLM}/scripts/session3_frontier.py', '/content/session3_frontier.py')
shutil.copy(f'{MASLD_LLM}/scripts/score_frontier.py', '/content/score_frontier.py')

import os
key = input('OpenRouter key (or press Enter to use Colab secret): ').strip()
if key:
    os.environ['OPENROUTER_API_KEY'] = key

In [ ]:
# run one model at a time (resumable). Verified-working roster:
%cd /content
!python3 session3_frontier.py moonshotai/kimi-k3 300

In [ ]:
# score everything -> leaderboard
!python3 score_frontier.py
import pandas as pd
df = pd.read_csv(f'{MASLD_LLM}/results/leaderboard.csv')
print(df.to_string(index=False))